# Security_Log_Analysis



# Use Case Description
Detecting anomalies within security logs is essential for effective operations. Given the massive volume of data and the difficulty of correlating disparate sources, manual review is often inefficient. Reasoning LLMs offer an automated solution, enabling the analysis of complex log streams to identify and flag security anomalies with greater precision.


## Model used for this use case
In this implementation, we utilized a Reasoning Model to analyze a unified set of user access logs. <br>
By correlating events including firewall activity, physical badge scans, and API login events, the model identifies behavioral deviations that indicate potential security threats.


## SetUp

We provide three different setup methods in this example. 

1. [Quickstart (Reasoning Model)](https://github.com/cisco-foundation-ai/cookbook/blob/main/1_quickstarts/Preview_Quickstart_reasoning_model.ipynb)
2. [Deploy on Sagemaker](https://github.com/cisco-foundation-ai/cookbook/blob/main/3_adoptions/deployment/sagemaker/foundation_sec_8b_reasoning/deploy.ipynb)
3. [Deploy with Baseten](https://github.com/cisco-foundation-ai/cookbook/tree/main/3_adoptions/deployment/baseten)

Please specify the preferred setup method in the cell below.

In [0]:
deployment_types = ["transformers", "sagemaker", "baseten"]

DEPLOYMENT_TYPE = ''  # Please input your preferred setup method from the list above

assert DEPLOYMENT_TYPE in deployment_types, "Value not found in list"

In [0]:
import json
import requests
from IPython.display import display, Markdown

MAX_TOKENS = 8192
TEMPERATURE = 0.3
# For Specific Setup Methods
## Transformers Setup
HUGGINGFACE_MODEL_ID = "fdtn-ai/Foundation-Sec-8B-Reasoning"  # Hugging Face model name. For reasoning model: "fdtn-ai/Foundation-Sec-8B-Reasoning"
## Sagemaker Setup
SAGEMAKER_ENDPOINT_NAME = ''  # Fill in your sagemaker deployment endpoint if applicable
## Baseten Setup
BASETEN_ENDPOINT_URL = ''  # Fill in your baseten deployment endpoint if applicable. Example: https://XXXXX.api.baseten.co/environments/production/sync/v1/chat/completions
BASETEN_API_KEY = ''  # Fill in your baseten API key if applicable


if DEPLOYMENT_TYPE == "transformers":
    # Transformers inference
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch
    import re

    def _get_device():
        if torch.cuda.is_available():
            return "cuda"
        elif torch.backends.mps.is_available():
            return "mps"
        else:
            return "cpu"

    DEVICE = _get_device()

    tokenizer = AutoTokenizer.from_pretrained(HUGGINGFACE_MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        pretrained_model_name_or_path=HUGGINGFACE_MODEL_ID,
        device_map="auto",
        torch_dtype=torch.float32,
    )

    generation_args = {
        "max_new_tokens": MAX_TOKENS,
        "temperature": TEMPERATURE,
        "do_sample": True,
        "use_cache": True,
        "eos_token_id": tokenizer.eos_token_id,
        "pad_token_id": tokenizer.pad_token_id,
    }

    def chat_generation(user_prompt):
        messages = [
            {"role": "user", "content": user_prompt},
        ]
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        # The old model version didn’t include the <think> token in the chat template.
        think_token = "<think>\n"
        if not inputs.endswith(think_token):
            inputs += think_token
        
        inputs = tokenizer(inputs, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                **generation_args,
            )
        response = tokenizer.decode(outputs[0], skip_special_tokens = False)

        # extract the reasoning part and the answer part
        match = re.search(r"<think>(.*?)</think>(.*)", response, re.DOTALL)

        if match:
            reasoning = match.group(1).strip().replace('<think>', '')
            answer = match.group(2).strip().replace('<|end_of_text|>', '')
        else:
            # Fallback in case tags are missing or malformed
            reasoning = ""
            answer = response.split("<|assistant|>")[-1].replace('<think>', '').replace('</think>', '').replace('<|end_of_text|>', '')
        
        return reasoning, answer

elif DEPLOYMENT_TYPE == "sagemaker":
    # Sagemaker inference
    import boto3
    sagemaker_runtime = boto3.client('sagemaker-runtime')
    endpoint_name = SAGEMAKER_ENDPOINT_NAME

    def chat_generation(user_prompt):
        messages = [
            {
                "role": "user",
                "content": user_prompt
            }
        ]
        payload = {
            "messages": messages,
            "temperature": TEMPERATURE,
            "max_tokens": MAX_TOKENS,
        }
        response = sagemaker_runtime.invoke_endpoint(
            EndpointName=endpoint_name,
            Body=json.dumps(payload),
            ContentType="application/json",
        )
        body = json.loads(response['Body'].read().decode('utf-8'))
        message = body["choices"][0]["message"]

        # The reasoning field name may vary by vLLM version
        reasoning = message.get("reasoning_content") or message.get("reasoning", "")
        answer = message.get("content", "")
        return reasoning, answer
else:
    # Baseten inference
    def chat_generation(user_prompt):
        data = {
            'messages': [{"role": "user", "content": user_prompt}],
            'max_tokens': MAX_TOKENS,
            'temperature': TEMPERATURE,
            }
        response = requests.post(
            BASETEN_ENDPOINT_URL,
            headers={"Authorization": f"Api-Key {BASETEN_API_KEY}"},
            json=data,
        )

        body = response.json()
        message = body["choices"][0]["message"]

        # The reasoning field name may vary by vLLM version
        reasoning = message.get("reasoning_content") or message.get("reasoning", "")
        answer = message.get("content", "")
        return reasoning, answer

## Log Events

Load the log events from the data/ folder

In [0]:
import pandas as pd

df = pd.read_csv('data/richard_access_events.csv')

log_events = df['_raw'].to_list()
LOG_COUNTS = len(log_events)

print(f"Loaded {LOG_COUNTS} events from richard_access_events.csv.\n")
print("-------Log events-------\n")
print(*log_events, sep='\n')

Loaded 22 events from richard_access_events.csv.

-------Log events-------

{"EventDate": "2021-08-20T16:19:09.000Z", "EventIdentifier": "b83d8e0d-2bfd-4e2f-a26e-43587f6621c3", "SourceIp": "135.84.144.251", "CreatedById": "0055e000005kMOsAAM", "Username": "richard@yellowtalon.co", "UserId": "0055e000002hyl3AAA", "RelatedEventIdentifier": null, "SessionKey": "uE2Y98IZLGim6kFX", "CreatedDate": "2021-08-20T16:19:09.941Z", "LoginKey": "JkPhSsy/cmFNWIqr", "SessionLevel": "STANDARD"}
{"EventDate": "2021-08-20T15:52:39.000Z", "AuthServiceId": null, "CountryIso": "US", "Platform": "Windows 10", "EvaluationTime": 112.0, "CipherSuite": "ECDHE-RSA-AES256-GCM-SHA384", "PostalCode": "45066", "ClientVersion": "N/A", "LoginGeoId": "04F5e00000NgPIZ", "LoginUrl": "frothly.secure.my.salesforce.com", "LoginHistoryId": "0Ya5e00000R9m7CCAR", "CreatedById": "0055e000005kMOsAAM", "SessionKey": null, "ApiType": "N/A", "AuthMethodReference": null, "LoginType": "Application", "PolicyOutcome": "Notified", "Statu

In [0]:
def make_prompt(log_events):
    return f"""You are a security expert in charge of analyzing security log events and detecting security risks.

Go over the list of access log events of a California-based employee Richard. 
Based on the log events, identify anomaly events based on unusual access patterns or login locations. 
Map security risks to Mitre Attack framework when possible.

## EVENTS
{str(log_events)}
"""

In [0]:
reasoning, answer = chat_generation(make_prompt(log_events))

print("=== Reasoning (chain-of-thought) ===")
display(Markdown(reasoning))
print()
print("=== Answer ===")
display(Markdown(answer))

=== Reasoning (chain-of-thought) ===



Alright, let me tackle this log analysis for Richard. First, I need to parse through all these events systematically. The logs show multiple entries around August 20, 2021, primarily from IP 135.84.144.251. 

Looking closely, there are two types of events here: Salesforce application logins and Check Point firewall/access logs. The Salesforce ones are straightforward—Richard logging into his account via browser. But the firewall logs reveal something more interesting. They show VPN tunnel establishment and SSL network extender usage. 

Hmm, the timestamps overlap between the Salesforce and firewall logs. Specifically, the first three firewall events (authorization, accept, log-in) occur exactly at 15:52:39, 15:51:56, and 15:50:34 respectively. Then another set at 15:48:08 and 15:45:23. All these times align with the Salesforce login attempts. 

Wait, the firewall logs mention "user=richards" while Salesforce uses "richard@yellowtalon.co". That's probably just different naming conventions. More importantly, the firewall logs show Richard accessing internal systems like "NetAccess" and "Browser" applications. 

Now, the real red flag emerges when looking at the geolocation data. While most Salesforce logins originate from Redwood City, CA (as expected), the firewall logs consistently show Richard connecting from Rosarito, Mexico. That's a significant geographical discrepancy! 

Further investigation reveals repeated access attempts to specific locations within ThirstyBerner Bar & Grille—Dock Bay 1, Brew Supply, Entrance. Each attempt shows identical risk scores of 100 and high speeds. This pattern strongly suggests unauthorized physical access attempts. 

Mapping this to MITRE ATT&CK: T1078 (Valid Accounts) applies since he's using legitimate credentials, but the location mismatch points to T1550 (Pass the Hash/Ticket) or possibly T1133 (External Remote Services). The physical access attempts align with T1098 (Account Manipulation) if someone compromised his badge. 

The risk score of 100 indicates critical severity. Immediate action is needed—revoking badges, changing passwords, and investigating the Mexico connection. This could indicate either an insider threat or credential compromise.



=== Answer ===




Based on the provided log events, I've identified several anomalies and mapped them to potential security risks using the MITRE ATT&CK framework:

### **Anomaly Events**
1. **Geographical Discrepancy**  
   - Multiple firewall/access logs show Richard accessing facilities in **Rosarito, Mexico**, while Salesforce logs place him in **Redwood City, CA**.  
   - *Risk*: Potential unauthorized physical access or credential misuse.  

2. **Repeated Access Attempts**  
   - Repeated badge-access events at **ThirstyBerner Bar & Grille** (Dock Bay 1, Brew Supply, Entrance) within minutes.  
   - *Risk*: High-frequency access may indicate tailgating, badge cloning, or insider threat activity.  

3. **VPN Tunnel Anomalies**  
   - Firewall logs show Richard establishing VPN tunnels to **SSL Network Extenders** during work hours.  
   - *Risk*: Could indicate exfiltration or lateral movement if combined with suspicious behavior.  

---

### **MITRE ATT&CK Mapping**  
| Event Type          | Technique ID       | Description                                                                 |
|---------------------|--------------------|-----------------------------------------------------------------------------|
| **Physical Access** | T1098 (Account Manipulation) | Unauthorized badge scans at Mexican facility sites.                         |
| **Credential Use**  | T1078 (Valid Accounts) | Legitimate credentials used from non-work locations.                          |
| **Network Activity**| T1572 (Protocol Tunneling) | VPN tunneling via SSL Network Extenders.                                    |

---

### **Recommended Actions**  
1. **Immediate Investigation**:  
   - Verify Richard’s presence in Mexico during the access attempts.  
   - Audit badge logs for cloned/forged badges.  

2. **Security Controls**:  
   - Enforce **multi-factor authentication (MFA)** for remote access.  
   - Implement **geofencing** for VPN connections.  
   - Review **physical security protocols** at ThirstyBerner Bar & Grille.  

3. **Forensic Analysis**:  
   - Correlate firewall/Salesforce logs to pinpoint the source of anomalous activity.  
   - Check for **unusual file transfers/exfiltration** during VPN sessions.  

> ⚠️ **Critical Risk Score**: All events carry a risk score of **100**, indicating immediate attention is required.